# Benard Convection with MIDL

Use the Benard dataset in this folder and run `MIDL` to discover dominant dimensionless groups.

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_notebook_dir = os.getcwd()
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '..', '..', '..'))
if project_root_dir not in sys.path:
    sys.path.append(project_root_dir)

import midl
importlib.reload(midl)
MIDL = midl.MIDL
calc_basis = midl.calc_basis


ModuleNotFoundError: No module named 'midl'

In [ ]:
# Load Benard dataset and define variables
df = pd.read_csv('dataset_rb.csv')
data = df.iloc[:, 2:12].to_numpy()

Nu = (data[:, 0] * data[:, 1]) / (data[:, 3] * data[:, 2])
Y = Nu

# h, delta_T, lambda, g, alpha, nu, kappa
X_raw = data[:, [1, 2, 3, 4, 5, 6, 7]]
variables = ['h', 'delta_T', 'lambda', 'g', 'alpha', 'nu', 'kappa']

D_in = np.array([
    [1, 0, 1, 1, 0, 2, 2],
    [0, 0, -3, -2, 0, -1, -1],
    [0, 0, 1, 0, 0, 0, 0],
    [0, 1, -1, 0, -1, 0, 0],
], dtype=float)

print('D_in rank:', np.linalg.matrix_rank(D_in))
print('D_in shape:', D_in.shape)


In [ ]:
# Compute basis and construct independent dimensionless groups Pi
basis, r = calc_basis(D_in)
print('Null space dimension:', D_in.shape[1] - r)
print('basis shape:', basis.shape)
print('basis:\n', basis)

Pi = np.exp(np.log(X_raw) @ basis)
print('Pi shape:', Pi.shape)


In [ ]:
# Run MIDL
model = MIDL(
    k_neighbors=6,
    de_maxiter=200,
    de_popsize=15,
    random_state=42,
)

result = model.fit(Pi_independent=Pi, pi_dependent=Y)
pi_hat = MIDL.compose_new_pi(Pi, result['W'])

print('\n=== MIDL Results ===')
print('MI scores:', result['mi_scores'])
print('dominant_q:', result['dominant_q'])
print('drop ratios I_i / I_(i+1):', result['drop_ratios'])

# Exponents in original variable space
alpha = basis @ result['W']
print('\n=== Recovered exponents in original variables ===')
print(alpha)


In [ ]:
# Plot: 2D always, and extra 3D when dominant_q >= 2
if result['dominant_q'] >= 2:
    ax2d, ax3d = MIDL.plot_component_vs_dependent(
        Pi_independent=Pi,
        pi_dependent=Y,
        W=result['W'],
        dominant_q=result['dominant_q'],
        component_index=0,
        title='Benard MIDL',
    )
else:
    ax2d = MIDL.plot_component_vs_dependent(
        Pi_independent=Pi,
        pi_dependent=Y,
        W=result['W'],
        dominant_q=result['dominant_q'],
        component_index=0,
        title='Benard MIDL',
    )

plt.show()
